In [0]:
from pyspark.sql.functions import avg

telemetry = spark.table("workspace.default.silver_telemetry")
machines = spark.table("workspace.default.silver_machines")

machine_baseline = (
    telemetry
    .join(machines, "machineID", "inner")
    .groupBy("machineID", "model", "age")
    .agg(
        avg("volt").alias("avg_voltage"),
        avg("rotate").alias("avg_rotation"),
        avg("pressure").alias("avg_pressure"),
        avg("vibration").alias("avg_vibration")
    )
    .orderBy("age", "model", "machineID")
)

machine_baseline.show(20, truncate=False)

+---------+------+---+------------------+------------------+------------------+------------------+
|machineID|model |age|avg_voltage       |avg_rotation      |avg_pressure      |avg_vibration     |
+---------+------+---+------------------+------------------+------------------+------------------+
|39       |model4|0  |170.6970263714612 |446.5840211310812 |100.75750038627487|40.23875293106218 |
|14       |model3|1  |170.81349949652855|446.5607206811618 |100.4160605907196 |40.376877075958056|
|28       |model4|1  |170.8251434508309 |446.20974026894095|100.5831427541017 |40.24977059682326 |
|81       |model4|1  |170.754384353638  |445.11428873283137|100.68614188304906|40.302762056838795|
|92       |model1|2  |170.60565651976566|447.04063416528146|101.44684430771468|40.26635774025729 |
|90       |model2|2  |170.9921268232851 |445.3417945420565 |101.31720147905403|40.281186577238486|
|5        |model3|2  |171.04443470435925|446.8585765042039 |101.06618841696496|40.264845827522535|
|86       

In [0]:
model_baseline = (
    machine_baseline
    .groupBy("model")
    .agg(
        avg("avg_voltage").alias("baseline_voltage"),
        avg("avg_rotation").alias("baseline_rotation"),
        avg("avg_pressure").alias("baseline_pressure"),
        avg("avg_vibration").alias("baseline_vibration"),
        avg("age").alias("average_age")
    )
    .orderBy("model")
)

model_baseline.show(truncate=False)

+------+------------------+------------------+------------------+------------------+------------------+
|model |baseline_voltage  |baseline_rotation |baseline_pressure |baseline_vibration|average_age       |
+------+------------------+------------------+------------------+------------------+------------------+
|model1|170.7834155073449 |446.373423227139  |101.2690597471576 |40.43766504054572 |12.25             |
|model2|170.7233116456343 |446.80877583256444|101.21643647276431|40.3916333433353  |12.764705882352942|
|model3|170.80560573709985|446.5127552687223 |100.66766078436008|40.39519997809593 |12.028571428571428|
|model4|170.77332789117045|446.71379670103926|100.6723217394926 |40.34400841861521 |9.34375           |
+------+------------------+------------------+------------------+------------------+------------------+



In [0]:
age_baseline = (
    machine_baseline
    .groupBy("age")
    .agg(
        avg("avg_voltage").alias("baseline_voltage"),
        avg("avg_rotation").alias("baseline_rotation"),
        avg("avg_pressure").alias("baseline_pressure"),
        avg("avg_vibration").alias("baseline_vibration")
    )
    .orderBy("age")
)

age_baseline.show(50, truncate=False)

+---+------------------+------------------+------------------+------------------+
|age|baseline_voltage  |baseline_rotation |baseline_pressure |baseline_vibration|
+---+------------------+------------------+------------------+------------------+
|0  |170.6970263714612 |446.5840211310812 |100.75750038627487|40.23875293106218 |
|1  |170.79767576699916|445.96158322764467|100.56178174262345|40.30980324320671 |
|2  |170.77091414019063|446.8213743844314 |100.99114426081792|40.254681998338   |
|3  |170.80900214396087|446.5503215899532 |100.86461026781987|40.25987057850583 |
|4  |170.81180355677563|446.58123726449304|100.82466506344167|40.281711846465896|
|5  |170.89489959110426|446.74699262036995|100.77410306305404|40.29300410794691 |
|6  |170.84097987562802|446.4000167566559 |100.75080169280925|40.272882065845494|
|7  |170.72011505304792|446.6895930494508 |100.75422237624895|40.3114935296716  |
|8  |170.65923480405507|446.5832379901162 |100.65311416397527|40.47537610412386 |
|9  |170.9035909

In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.gold_q2_machine_baseline"

if not spark.catalog.tableExists(target_table):

    machine_baseline.write.format("delta").saveAsTable(target_table)

else:

    target = DeltaTable.forName(spark, target_table)

    target.alias("t").merge(
        machine_baseline.alias("s"),
        "t.machineID = s.machineID"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print("Gold Q2 machine baseline table created/updated")
machine_baseline.printSchema()

Gold Q2 machine baseline table created/updated
root
 |-- machineID: integer (nullable = true)
 |-- model: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- avg_voltage: double (nullable = true)
 |-- avg_rotation: double (nullable = true)
 |-- avg_pressure: double (nullable = true)
 |-- avg_vibration: double (nullable = true)



In [0]:
display(age_baseline)

age,baseline_voltage,baseline_rotation,baseline_pressure,baseline_vibration
0,170.6970263714612,446.5840211310812,100.75750038627487,40.23875293106218
1,170.79767576699916,445.96158322764467,100.56178174262345,40.30980324320671
2,170.77091414019063,446.8213743844314,100.99114426081792,40.254681998338
3,170.80900214396087,446.5503215899532,100.86461026781987,40.25987057850583
4,170.81180355677563,446.58123726449304,100.82466506344167,40.281711846465896
5,170.89489959110426,446.74699262036995,100.77410306305404,40.29300410794691
6,170.84097987562802,446.4000167566559,100.75080169280925,40.272882065845494
7,170.72011505304792,446.6895930494508,100.75422237624895,40.3114935296716
8,170.65923480405507,446.5832379901162,100.65311416397527,40.47537610412386
9,170.90359096362573,446.70931045281,100.67709072229677,40.261576813439206


Databricks visualization. Run in Databricks to view.

In [0]:
display(model_baseline)

model,baseline_voltage,baseline_rotation,baseline_pressure,baseline_vibration,average_age
model1,170.7834155073449,446.373423227139,101.2690597471576,40.43766504054572,12.25
model2,170.7233116456343,446.80877583256444,101.21643647276431,40.3916333433353,12.764705882352942
model3,170.80560573709985,446.5127552687223,100.66766078436008,40.39519997809593,12.028571428571428
model4,170.77332789117045,446.71379670103926,100.6723217394926,40.34400841861521,9.34375


Databricks visualization. Run in Databricks to view.

In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.gold_q2_model_baseline"

if not spark.catalog.tableExists(target_table):

    model_baseline.write.format("delta").saveAsTable(target_table)

else:

    target = DeltaTable.forName(spark, target_table)

    target.alias("t").merge(
        model_baseline.alias("s"),
        "t.model = s.model"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print("Gold Q2 model baseline table created/updated")
model_baseline.printSchema()

Gold Q2 model baseline table created/updated
root
 |-- model: string (nullable = true)
 |-- baseline_voltage: double (nullable = true)
 |-- baseline_rotation: double (nullable = true)
 |-- baseline_pressure: double (nullable = true)
 |-- baseline_vibration: double (nullable = true)
 |-- average_age: double (nullable = true)

